# 🌳 Tree-Ring Watermarks — Colab Demo

This notebook demonstrates:
1. **Baseline** (Wen et al. 2023) — inject a Fourier ring into latent channel 0
2. **MultiBitWatermark** — encode a binary payload in Fourier phase angles
3. **LogPolarWatermark** — rotation-invariant watermarking via log-polar FFT
4. **EnsembleWatermark** — embed rings in all 4 latent channels, detect by majority vote

**Target hardware**: Google Colab Free Tier T4 GPU  
**Runtime**: ~15 minutes for a 5-image quick demo

> ⚠️ Make sure you have selected **Runtime → Change runtime type → T4 GPU** before running.

## 1. Install requirements

In [ ]:
%%capture
!pip install diffusers==0.21.4 transformers==4.48.0 accelerate==0.24.1 \
             torch torchvision scipy scikit-image matplotlib tqdm Pillow

## 2. Clone repository

In [ ]:
import os

REPO_URL = "https://github.com/Ashhad1130/Invisible_Watermark_GNN.git"
REPO_DIR = "Invisible_Watermark_GNN"

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL}

os.chdir(REPO_DIR)
print("Working directory:", os.getcwd())

## 3. Imports & configuration

In [ ]:
import torch
from diffusers import DDIMScheduler, StableDiffusionPipeline
import matplotlib.pyplot as plt
import numpy as np

from config import Config
from watermark import TreeRingWatermark, MultiBitWatermark, LogPolarWatermark, EnsembleWatermark
from utils.attacks import crop, rotate, jpeg, gaussian_blur, gaussian_noise
from utils.metrics import tpr, fpr, balanced_accuracy, ber

# Configuration
cfg = Config(
    device="cuda" if torch.cuda.is_available() else "cpu",
    num_inference_steps=20,   # fewer steps for speed in demo
    guidance_scale=7.5,
)
print(f"Device: {cfg.device}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 4. Load Stable Diffusion v1.5 pipeline

In [ ]:
print("Loading SD v1.5 pipeline (this may take a few minutes on first run)…")
pipe = StableDiffusionPipeline.from_pretrained(
    cfg.model_id,
    torch_dtype=torch.float16 if cfg.device == "cuda" else torch.float32,
    safety_checker=None,
)
pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)
pipe = pipe.to(cfg.device)
pipe.set_progress_bar_config(disable=True)
print("Pipeline ready!")

## 5. Define demo prompts

In [ ]:
PROMPTS = [
    "a serene mountain landscape at sunset, photorealistic",
    "a futuristic city skyline with neon lights, digital art",
    "a bouquet of wildflowers in a glass vase, oil painting",
    "an astronaut floating in deep space, cinematic",
    "a golden retriever playing in autumn leaves, photography",
]
N_IMAGES = 5  # use all 5 prompts
prompts = PROMPTS[:N_IMAGES]

## 6. Baseline – Tree-Ring Watermark (Wen et al. 2023)

In [ ]:
baseline = TreeRingWatermark(cfg, pipe)

prompt = prompts[0]
print(f"Generating watermarked image for: '{prompt}'")
img_wm, latent_wm = baseline.generate_watermarked(prompt, seed=42, return_latents=True)
img_clean = baseline.generate_unwatermarked(prompt, seed=42)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(img_wm); axes[0].set_title("Watermarked"); axes[0].axis("off")
axes[1].imshow(img_clean); axes[1].set_title("Clean (no watermark)"); axes[1].axis("off")
plt.suptitle("Baseline: Tree-Ring Watermark")
plt.tight_layout()
plt.show()

# Detection
is_wm, score_wm = baseline.detect(img_wm)
is_cl, score_cl = baseline.detect(img_clean)
print(f"Watermarked image → detected={is_wm}, score={score_wm:.3f}")
print(f"Clean image       → detected={is_cl}, score={score_cl:.3f}")

## 7. Robustness under attacks

In [ ]:
attacks = {
    "No attack":       lambda img: img,
    "Crop 80%":        lambda img: crop(img, 0.8),
    "Rotate 45°":      lambda img: rotate(img, 45),
    "JPEG Q=50":       lambda img: jpeg(img, 50),
    "Gaussian Blur 2": lambda img: gaussian_blur(img, 2.0),
    "Noise σ=0.05":    lambda img: gaussian_noise(img, 0.05),
}

fig, axes = plt.subplots(1, len(attacks), figsize=(18, 3))
scores = []
for ax, (name, fn) in zip(axes, attacks.items()):
    attacked = fn(img_wm)
    _, score = baseline.detect(attacked)
    scores.append(score)
    ax.imshow(attacked)
    ax.set_title(f"{name}\n score={score:.3f}", fontsize=8)
    ax.axis("off")
plt.suptitle("Baseline watermark under attacks")
plt.tight_layout()
plt.show()
print("Scores:", {k: f"{s:.3f}" for k, s in zip(attacks.keys(), scores)})

## 8. Novel Method 1 – MultiBitWatermark

In [ ]:
import numpy as np

multibit = MultiBitWatermark(cfg, pipe)
payload = np.array([1,0,1,1,0,0,1,0,1,1,0,1,0,0,1,0], dtype=np.uint8)
print(f"Payload: {payload}")

img_mb, _ = multibit.generate_watermarked(prompts[1], payload=payload, seed=7, return_latents=True)
is_mb, score_mb, decoded = multibit.detect(img_mb)

from utils.metrics import ber as _ber
print(f"Detected: {is_mb}, score: {score_mb:.3f}")
print(f"Original : {payload}")
print(f"Decoded  : {decoded}")
print(f"BER      : {_ber(payload, decoded):.3f}")

plt.imshow(img_mb)
plt.title(f"MultiBit Watermark  (BER={_ber(payload, decoded):.3f})")
plt.axis("off")
plt.show()

## 9. Novel Method 2 – LogPolarWatermark (rotation-invariant)

In [ ]:
logpolar = LogPolarWatermark(cfg, pipe)
img_lp, _ = logpolar.generate_watermarked(prompts[2], seed=13, return_latents=True)

rotation_angles = [0, 15, 45, 90]
fig, axes = plt.subplots(1, len(rotation_angles), figsize=(14, 4))
for ax, angle in zip(axes, rotation_angles):
    rotated = rotate(img_lp, angle)
    _, score = logpolar.detect(rotated)
    ax.imshow(rotated)
    ax.set_title(f"Rot {angle}°\nscore={score:.3f}", fontsize=9)
    ax.axis("off")
plt.suptitle("LogPolar Watermark – Rotation Invariance")
plt.tight_layout()
plt.show()

## 10. Novel Method 3 – EnsembleWatermark (4-channel majority vote)

In [ ]:
ensemble = EnsembleWatermark(cfg, pipe)
img_ens, _ = ensemble.generate_watermarked(prompts[3], seed=21, return_latents=True)

is_ens, score_ens, ch_scores = ensemble.detect(img_ens)
print(f"Detected: {is_ens}, overall score: {score_ens:.2f}")
print(f"Per-channel scores: {[f'{s:.3f}' for s in ch_scores]}")

# Show robustness
for attack_name, attack_fn in [("No attack", lambda i: i), 
                                ("JPEG Q=50", lambda i: jpeg(i, 50)),
                                ("Crop 80%", lambda i: crop(i, 0.8))]:
    attacked = attack_fn(img_ens)
    is_d, score_d, _ = ensemble.detect(attacked)
    print(f"  {attack_name:<15} → detected={is_d}, score={score_d:.2f}")

## 11. Quick 5-image benchmark summary

In [ ]:
import time

METHODS = {
    "Baseline":  TreeRingWatermark(cfg, pipe),
    "MultiBit":  MultiBitWatermark(cfg, pipe),
    "LogPolar":  LogPolarWatermark(cfg, pipe),
    "Ensemble":  EnsembleWatermark(cfg, pipe),
}

QUICK_ATTACKS = {
    "none":      lambda img: img,
    "jpeg_50":   lambda img: jpeg(img, 50),
    "rotate_45": lambda img: rotate(img, 45),
}

print(f"{'Method':<14} {'Attack':<14} {'TPR':>6} {'FPR':>6} {'BAcc':>6}")
print("-" * 48)

for method_name, wm in METHODS.items():
    for attack_name, attack_fn in QUICK_ATTACKS.items():
        y_true, y_pred = [], []
        for i, prompt in enumerate(prompts):
            seed = i * 10
            if isinstance(wm, MultiBitWatermark):
                img_w, _ = wm.generate_watermarked(prompt, seed=seed, return_latents=True)
                attacked_w = attack_fn(img_w)
                is_w, _, _ = wm.detect(attacked_w)
            elif isinstance(wm, EnsembleWatermark):
                img_w, _ = wm.generate_watermarked(prompt, seed=seed, return_latents=True)
                attacked_w = attack_fn(img_w)
                is_w, _, _ = wm.detect(attacked_w)
            else:
                img_w, _ = wm.generate_watermarked(prompt, seed=seed, return_latents=True)
                attacked_w = attack_fn(img_w)
                is_w, _ = wm.detect(attacked_w)

            gen = torch.Generator(device=cfg.device).manual_seed(seed + 500)
            img_c = pipe(
                prompt=prompt, generator=gen,
                num_inference_steps=cfg.num_inference_steps,
                guidance_scale=cfg.guidance_scale,
                output_type="pil",
            ).images[0]
            attacked_c = attack_fn(img_c)
            if isinstance(wm, MultiBitWatermark):
                is_c, _, _ = wm.detect(attacked_c)
            elif isinstance(wm, EnsembleWatermark):
                is_c, _, _ = wm.detect(attacked_c)
            else:
                is_c, _ = wm.detect(attacked_c)

            y_true += [True, False]
            y_pred += [is_w, is_c]

        _tpr = tpr(y_true, y_pred)
        _fpr = fpr(y_true, y_pred)
        _ba  = balanced_accuracy(y_true, y_pred)
        print(f"{method_name:<14} {attack_name:<14} {_tpr:>6.3f} {_fpr:>6.3f} {_ba:>6.3f}")

print("\n✅ Demo complete!")

## 12. (Optional) Full evaluation + analysis

Run the full pipeline and generate publication-ready figures.

In [ ]:
# Uncomment to run full evaluation (takes ~30-60 min on T4)
# !python evaluate.py --device cuda --n_images 5 --fp16
# !python analyse.py
# from IPython.display import Image
# Image("results/figures/heatmap.png")